In [ ]:
# LEAK HUNT: do DICOM metadata tags reveal the 12 findings (readable at TEST time -> cheap 0.9+)?
import os,glob,pydicom,pandas as pd,numpy as np,collections
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/train.csv',recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base',base); print(os.listdir(base))
tr=pd.read_csv(os.path.join(base,'train.csv'))
LAB=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
gold=tr[tr['ACL'].notna()]
print('gold labeled studies:',len(gold))

In [ ]:
# dump ALL non-pixel DICOM tags for a few gold studies (with known labels) + look for finding text
ts=glob.glob(os.path.join(base,'train_series','*'))
print('train_series dirs:',len(ts))
def first_dcm(study):
    d=glob.glob(os.path.join(base,'train_series',study,'*','*.dcm'))
    return d[0] if d else None
# sample gold studies spanning positives
sample=gold.head(8)
for _,row in sample.iterrows():
    sid=row['StudyInstanceUID']; f=first_dcm(sid)
    if not f: print(sid,'NO DCM'); continue
    dc=pydicom.dcmread(f, stop_before_pixels=True)
    poslabels=[l for l in LAB if row[l]==1]
    print('
==== study',sid[:20],'POS:',poslabels,'====')
    for el in dc:
        if el.keyword in ('PixelData',): continue
        v=str(el.value)
        if len(v)>120: v=v[:120]+'...'
        print(f'  ({el.tag}) {el.keyword or el.name}: {v}')

In [ ]:
# scan text-bearing tags across MANY studies for correlation with labels
import re
def txt_tags(study):
    f=first_dcm(study)
    if not f: return {}
    dc=pydicom.dcmread(f, stop_before_pixels=True)
    out={}
    for el in dc:
        if el.VR in ('LO','LT','SH','ST','UT','PN','CS','DA','TM') or 'Description' in (el.keyword or '') or 'Comment' in (el.keyword or ''):
            out[el.keyword or str(el.tag)]=str(el.value)
    return out
allt=collections.defaultdict(set)
for _,row in gold.iterrows():
    for k,v in txt_tags(row['StudyInstanceUID']).items():
        allt[k].add(v)
print('distinct values per text tag (gold studies):')
for k,vs in sorted(allt.items(), key=lambda x:-len(x[1])):
    ex=list(vs)[:5]
    print(f'  {k}: {len(vs)} distinct  e.g. {ex}')

In [ ]:
# also dump the placeholder TEST dicom tags (are they the same structure? any finding text?)
tser=glob.glob(os.path.join(base,'test_series','*'))
print('test_series dirs:',len(tser))
for sd in tser[:3]:
    d=glob.glob(os.path.join(sd,'*','*.dcm'))
    if not d: print(sd,'no dcm'); continue
    dc=pydicom.dcmread(d[0], stop_before_pixels=True)
    print('
==== TEST',os.path.basename(sd)[:20],'====')
    for el in dc:
        if el.keyword=='PixelData': continue
        v=str(el.value); v=v[:120]+'...' if len(v)>120 else v
        print(f'  {el.keyword or el.name}: {v}')